# Notebook 033: Delta Table Creation, Metadata Inspection and Transaction Log Internals

**Module:** Module 04: Delta Lake Architecture, ACID Operations & Time Travel  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 03: Databricks Architecture & Video 14: Master Unity Catalog  

---

## 1. Business Scenario & Real-World Context
A lakehouse platform architect requires an immutable, audit-compliant storage layer for financial ledger records. Standard Parquet files lack ACID atomicity and suffer from partial write failures during cluster spot instance terminations.

## 2. Technical Requirement & Problem Statement
Create a Delta Lake table with explicit schema, inspect its physical storage directory structure, read the JSON commit log files inside `_delta_log/`, parse checkpoint parity files, and query table details via `DESCRIBE DETAIL` and `DESCRIBE HISTORY`.

## 3. Spark & Databricks Architecture Notes
Delta Lake brings ACID transactions to Apache Spark using a write-ahead log stored in `_delta_log/`. Every transaction creates an atomic JSON commit (e.g. `000000.json`). Every 10 commits, Delta writes a consolidated Parquet checkpoint file to bound state reconstruction time.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_033_Delta_Table_Creation,_Metadata_Inspection_and_Transaction_Log_Internals")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Creating a Managed Delta Table on Local Storage
Write initial transactions to a local Delta table path.

In [ ]:
import shutil
from pathlib import Path

delta_path = "/tmp/delta_ledger_033"
if Path(delta_path).exists():
    shutil.rmtree(delta_path)

ledger_data = [
    ("TX-001", "ACC-101", 1500.0, "DEPOSIT", "2023-11-20 10:00:00"),
    ("TX-002", "ACC-102", 450.0, "WITHDRAWAL", "2023-11-20 10:05:00"),
    ("TX-003", "ACC-101", 200.0, "TRANSFER", "2023-11-20 10:15:00")
]

schema = "txn_id STRING, account_id STRING, amount DOUBLE, txn_type STRING, created_at STRING"
df_ledger = spark.createDataFrame(ledger_data, schema)

# Write as Delta format
df_ledger.write.format("delta").mode("overwrite").save(delta_path)
print(f"Delta table created at: {delta_path}")

## 6. Inspecting Physical Delta Directory & _delta_log JSON Commits
Examine the commit log files generated by the transaction.

In [ ]:
delta_dir = Path(delta_path)
log_dir = delta_dir / "_delta_log"

print(f"Log directory exists: {log_dir.exists()}")
log_files = sorted(list(log_dir.glob("*.json")))
for lf in log_files:
    print(f"Commit file found: {lf.name}")
    with open(lf, "r") as f:
        # Show first 3 action lines in the JSON log
        lines = [f.readline().strip() for _ in range(3)]
        print(f"  Actions snippet: {lines[:2]}")

## 7. Querying Table Details & History Metadata
Use SQL metadata commands to inspect table format, partition columns, and version.

In [ ]:
# Query metadata via SQL
df_detail = spark.sql(f"DESCRIBE DETAIL delta.`{delta_path}`")
df_detail.select("format", "id", "name", "numFiles", "sizeInBytes").show(truncate=False)

df_history = spark.sql(f"DESCRIBE HISTORY delta.`{delta_path}`")
df_history.select("version", "timestamp", "userId", "operation", "operationParameters").show(truncate=False)

## 9. Key Architectural Takeaways
- **Delta Lake stores data as standard Parquet files paired with an ACID transaction log in `_delta_log/`.**
- **Atomicity is guaranteed via mutual exclusion protocols (e.g. Put-If-Absent) in cloud object stores.**
- **`DESCRIBE DETAIL` and `DESCRIBE HISTORY` provide instant visibility into table health, file count, and commit activity.**

## 10. Senior Data Engineering Interview Questions
1. How does Delta Lake guarantee ACID transactions over eventually consistent object storage like S3 or ADLS?
2. What is the purpose of Delta checkpoint Parquet files created every 10 commits?
3. What information is stored inside each `_delta_log/00000X.json` commit file?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
